# exploratory analysis of air quality and weather

this notebook looks at the hourly data the pipeline collected for 63 points in Viet Nam, one per former province. it covers how complete the data is, where and when the air is polluted, how the weather drives PM2.5 and what that means for the forecast models.

the data comes from the serving layer. every query goes to Trino, the same path the dashboards use, and each result is kept as a parquet file under `data/eda`, so the notebook also runs while the cluster is stopped. the charts are written to `docs/images` with the prefix `eda`. the comments between the cells describe the extract of 2026-09-01 to 2026-11-11.

In [ ]:
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.ticker import PercentFormatter, StrMethodFormatter

from airpulse.analysis import eda, plots
from airpulse.analysis.trino_io import FrameStore
from airpulse.common.aqi import CATEGORIES
from airpulse.common.config import REPO_ROOT, load_config

cfg = load_config()
# results are read from data/eda when a copy exists, REFRESH sends every query to trino again.
REFRESH = False
store = FrameStore(cfg, REPO_ROOT / "data" / "eda", refresh=REFRESH)
FIGURES = REPO_ROOT / "docs" / "images"
HIVE = f"{cfg.trino.hive_catalog}.{cfg.trino.schema_name}"
VIEWS = f"{HIVE}_views"
REGIONS = ["north", "central", "south"]
PM = "PM2.5, µg/m³"
DAY_FORMAT = mdates.DateFormatter("%m-%d")
plots.apply_style()

## the data set

the clean layer holds one row per point and hour. air quality comes from the CAMS model and weather from the forecast models, both served by Open-Meteo, so every row is a model value for the coordinates of a point and not a station reading.

In [ ]:
locations = store.load(
    "locations", f"select location_id, name, lat, lon, region, province from {HIVE}.ref_locations"
)
air = store.load(
    "air_hourly",
    f"""
    select location_id, event_ts_utc, pm2_5_ugm3, pm10_ugm3, no2_ugm3, so2_ugm3, o3_ugm3, co_ugm3,
           aqi_us, aqi_computed, aqi_dominant, aqi_category
    from {HIVE}.clean_air_hourly
    """,
)
weather = store.load(
    "weather_hourly",
    f"""
    select location_id, event_ts_utc, temperature_c, humidity_pct, precipitation_mm,
           pressure_msl_hpa, cloud_cover_pct, wind_speed_kmh, wind_gusts_kmh
    from {HIVE}.clean_weather_hourly
    """,
)
hourly = air.merge(weather, on=["location_id", "event_ts_utc"]).merge(
    locations[["location_id", "name", "region"]], on="location_id"
)
hourly = eda.with_local_time(hourly, cfg.timezone)
print(len(air), "air rows and", len(weather), "weather rows for", air["location_id"].nunique(), "points")
print("from", air["event_ts_utc"].min(), "to", air["event_ts_utc"].max(), "UTC")

In [ ]:
coverage = eda.coverage_by_day(air, len(locations))
partial = coverage[coverage["share"] < 1]
print(len(coverage), "days,", len(coverage) - len(partial), "of them complete")
display(partial)
print("the largest share of missing values in a column is", hourly.isna().mean().max())

every point has every hour and no value is missing. that is a property of model data, a station network would show gaps. the last day is partial because the extract was taken during that day.

the raw layer looks different. the producer fetches the last three hours again on every cycle to pick up corrections, and a backfill can overlap with the live stream or with an earlier attempt, so one event is stored several times. the batch cleaning keeps the copy that was ingested last.

In [ ]:
raw_counts = store.load(
    "raw_counts",
    f"""
    select 'air' as domain, dt, count(*) as raw_rows, count(distinct event_id) as events
    from {HIVE}.raw_air_hourly group by dt
    union all
    select 'weather' as domain, dt, count(*) as raw_rows, count(distinct event_id) as events
    from {HIVE}.raw_weather_hourly group by dt
    """,
)
raw_air = raw_counts[raw_counts["domain"] == "air"].copy()
raw_air["day"] = pd.to_datetime(raw_air["dt"])
raw_air["copies"] = raw_air["raw_rows"] - raw_air["events"]
raw_air = raw_air.sort_values("day")

fig, ax = plt.subplots()
bar = {"width": 0.8, "edgecolor": plots.SURFACE, "linewidth": 0.8}
ax.bar(raw_air["day"], raw_air["events"], color=plots.SERIES[0], label="distinct events", **bar)
ax.bar(
    raw_air["day"],
    raw_air["copies"],
    bottom=raw_air["events"],
    color=plots.SERIES[1],
    label="repeated copies",
    **bar,
)
ax.set_title("rows per day in the raw air quality layer")
ax.set_ylabel("rows")
ax.yaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))
ax.xaxis.set_major_formatter(DAY_FORMAT)
ax.legend(loc="upper left", ncols=2)
plots.save_figure(fig, FIGURES, "eda_raw_rows_per_day")
plt.show()

totals = raw_counts.groupby("domain")[["raw_rows", "events"]].sum()
totals["clean_rows"] = [len(air), len(weather)]
display(totals)

the copies have two origins. the first backfill stopped on a rate limit after it had stored 30 points for 2026-09 and the second attempt loaded the whole range again, so those points have two copies for that month. the days with a live stream hold several copies of an event, up to twelve on 2026-11-07, one for every producer cycle that still had the hour in its window and one for each gap fill. the clean layer has exactly one row per event in both domains.

### points that share a grid cell

the air quality model has a coarser grid than the weather models. points that fall in the same cell receive the same series, which matters when points are counted or averaged.

In [ ]:
cells = eda.shared_series(air, "pm2_5_ugm3").merge(
    locations[["location_id", "name", "region"]], on="location_id"
)
shared = cells[cells["cell_size"] > 1]
print(cells["cell"].nunique(), "distinct PM2.5 series for", len(cells), "points")
print(eda.shared_series(weather, "temperature_c")["cell"].nunique(), "distinct temperature series")
display(shared.groupby("cell").agg(region=("region", "first"), points=("name", ", ".join)))

sixteen points sit in six shared cells, among them Ha Noi with Bac Ninh and Bac Giang, and Ho Chi Minh with Thu Dau Mot and Bien Hoa. the 63 points carry 53 independent air quality series, while every point has its own weather series. the regional figures below still use every point, since a province is the unit of interest, but pollution episodes are counted once per cell.

## how polluted the air is

In [ ]:
summary = hourly.groupby("region")["pm2_5_ugm3"].describe(percentiles=[0.5, 0.9, 0.99])
display(summary.loc[REGIONS].round(1))

In [ ]:
# daily means use local days with all 24 hours, the first and the last day of the extract are cut.
hours_per_day = hourly.groupby(["location_id", "day_local"])["hour_local"].transform("nunique")
daily = (
    hourly[hours_per_day == 24]
    .groupby(["region", "location_id", "name", "day_local"], as_index=False)
    .agg(
        pm2_5=("pm2_5_ugm3", "mean"),
        wind_speed=("wind_speed_kmh", "mean"),
        precipitation=("precipitation_mm", "sum"),
        temperature=("temperature_c", "mean"),
    )
)
limits = {
    "WHO guideline": cfg.aqi.pm2_5_daily_limit_who,
    "QCVN limit": cfg.aqi.pm2_5_daily_limit_qcvn,
}

fig, ax = plt.subplots()
for region in REGIONS:
    values = np.sort(daily.loc[daily["region"] == region, "pm2_5"])
    share = np.arange(1, len(values) + 1) / len(values)
    ax.plot(values, share, label=region, color=plots.REGION_COLORS[region])
for name, limit in limits.items():
    plots.reference_line(ax, limit, f"{name} {limit:g}", vertical=True)
ax.set_xlim(0, daily["pm2_5"].quantile(0.995))
ax.set_ylim(0, 1)
ax.yaxis.set_major_formatter(PercentFormatter(1))
ax.set_title("share of point days at or below a daily mean PM2.5")
ax.set_xlabel(f"daily mean {PM}")
ax.legend(loc="center right")
plots.save_figure(fig, FIGURES, "eda_daily_pm2_5_distribution")
plt.show()

exceedance = eda.exceedance_share(
    daily,
    "pm2_5",
    {"above_who": limits["WHO guideline"], "above_qcvn": limits["QCVN limit"]},
    "region",
)
display(exceedance.loc[REGIONS].round(3))

In [ ]:
order = [name for _, name in CATEGORIES]
shares = pd.crosstab(hourly["region"], hourly["aqi_category"], normalize="index")
shares = shares.reindex(index=REGIONS, columns=order, fill_value=0.0)
# the two highest classes are rare, they share the darkest step of the ramp.
shares["very_unhealthy_or_hazardous"] = shares.pop("very_unhealthy") + shares.pop("hazardous")

fig, ax = plt.subplots(figsize=(8.0, 2.8))
left = np.zeros(len(shares))
for step, (column, color) in enumerate(zip(shares.columns, plots.ORDINAL)):
    ax.barh(
        shares.index,
        shares[column],
        left=left,
        height=0.55,
        color=color,
        edgecolor=plots.SURFACE,
        linewidth=2,
        label=column.replace("_", " "),
    )
    for row, value in enumerate(shares[column]):
        if value >= 0.06:
            ink = plots.INK if step < 2 else plots.SURFACE
            ax.text(left[row] + 0.01, row, f"{value:.0%}", va="center", fontsize=9, color=ink)
    left += shares[column].to_numpy()
ax.invert_yaxis()
ax.set_xlim(0, 1)
ax.xaxis.set_major_formatter(PercentFormatter(1))
ax.grid(False)
ax.set_title("share of hours in each AQI class")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.15), ncols=3)
plots.save_figure(fig, FIGURES, "eda_aqi_classes_by_region")
plt.show()
display(shares.round(3))

In [ ]:
by_point = (
    hourly.groupby(["location_id", "name", "region"], as_index=False)["pm2_5_ugm3"]
    .mean()
    .merge(locations[["location_id", "lat", "lon"]], on="location_id")
)
# name and label offset in points, the labels go to the side with no neighbors.
labeled = {"Ha Noi": (48, 22), "Da Nang": (10, 0), "Ho Chi Minh": (36, -30)}

fig, ax = plt.subplots(figsize=(4.8, 7.0))
points = ax.scatter(
    by_point["lon"],
    by_point["lat"],
    c=by_point["pm2_5_ugm3"],
    cmap=plots.SEQUENTIAL,
    s=70,
    edgecolor=plots.SURFACE,
    linewidth=1.5,
)
for name, offset in labeled.items():
    city = by_point[by_point["name"] == name].iloc[0]
    ax.annotate(
        name,
        xy=(city["lon"], city["lat"]),
        xytext=offset,
        textcoords="offset points",
        va="center",
        fontsize=9,
        arrowprops={"arrowstyle": "-", "color": plots.MUTED_INK, "linewidth": 0.8},
    )
ax.set_aspect("equal")
ax.grid(False)
ax.set_xlabel("longitude")
ax.set_ylabel("latitude")
ax.set_title("mean PM2.5 by point")
fig.colorbar(points, ax=ax, shrink=0.5, label=f"mean {PM}")
plots.save_figure(fig, FIGURES, "eda_mean_pm2_5_map")
plt.show()

ranked = by_point.sort_values("pm2_5_ugm3", ascending=False)[["name", "region", "pm2_5_ugm3"]]
display(pd.concat([ranked.head(8), ranked.tail(5)]).round(1))

the north is the most polluted region by a wide margin. its mean PM2.5 of 30 µg/m³ is more than twice the central mean and one and a half times the southern one. three of four point days in the north are above the WHO guideline of 15 and one in six is above the national limit of 50, against 2 percent in the south and 1 percent in the center. the hourly index tells the same story, the north spends 19 percent of its hours in the unhealthy classes, the other two regions 3 and 4 percent.

the map shows where this comes from. the Red River delta forms one block of high values, from Hung Yen with a mean of 48.5 to Ha Noi with 43.7, while the central highlands around Da Lat, Kon Tum and Pleiku stay near 7.

## when the air is polluted

In [ ]:
by_day = daily.groupby(["day_local", "region"])["pm2_5"].mean().unstack()[REGIONS]

fig, ax = plt.subplots()
for region in REGIONS:
    ax.plot(by_day.index, by_day[region], label=region, color=plots.REGION_COLORS[region])
for name, limit in limits.items():
    plots.reference_line(ax, limit, f"{name} {limit:g}")
ax.set_ylim(0)
ax.xaxis.set_major_formatter(DAY_FORMAT)
ax.set_title("daily mean PM2.5 by region")
ax.set_ylabel(PM)
plots.legend_above(ax)
plots.save_figure(fig, FIGURES, "eda_daily_pm2_5_by_region")
plt.show()
display(by_day.describe().loc[["min", "50%", "max"]].round(1))

In [ ]:
profile = eda.diurnal_profile(hourly, "pm2_5_ugm3", "region")[REGIONS]

fig, ax = plt.subplots()
for region in REGIONS:
    ax.plot(profile.index, profile[region], label=region, color=plots.REGION_COLORS[region])
plots.label_line_ends(ax)
ax.set_xticks(range(0, 24, 3))
ax.set_xlim(0, 25.5)
ax.set_ylim(0)
ax.set_title("mean PM2.5 by hour of the day")
ax.set_xlabel("local hour")
ax.set_ylabel(PM)
plots.legend_above(ax)
plots.save_figure(fig, FIGURES, "eda_diurnal_pm2_5")
plt.show()
display(profile.loc[[3, 6, 9, 12, 15, 18, 21]].round(1))

In [ ]:
north = hourly[(hourly["region"] == "north") & (hours_per_day == 24)]
grid = north.pivot_table(index="hour_local", columns="day_local", values="pm2_5_ugm3")

fig, ax = plt.subplots(figsize=(9.0, 3.4))
mesh = ax.pcolormesh(
    grid.columns, grid.index, grid.to_numpy(), cmap=plots.SEQUENTIAL, shading="nearest"
)
ax.grid(False)
ax.set_yticks(range(0, 24, 6))
ax.set_ylabel("local hour")
ax.xaxis.set_major_formatter(DAY_FORMAT)
ax.set_title("PM2.5 in the north by day and hour")
fig.colorbar(mesh, ax=ax, label=PM, pad=0.02)
plots.save_figure(fig, FIGURES, "eda_north_pm2_5_day_hour")
plt.show()

In [ ]:
one_per_cell = hourly[hourly["location_id"].isin(cells["cell"])]
runs = eda.episodes(one_per_cell, "aqi_computed", cfg.alerts.aqi_level, min_hours=6).merge(
    locations[["location_id", "name", "region"]], on="location_id"
)
print(len(runs), "episodes of six hours or more above an AQI of", cfg.alerts.aqi_level)
by_region = runs.groupby("region").agg(
    episodes=("hours", "size"), hours=("hours", "sum"), longest=("hours", "max")
)
display(by_region.reindex(REGIONS))
display(runs.sort_values("hours", ascending=False)[["name", "region", "start", "end", "hours", "peak"]].head(8))

pollution in the north comes in episodes that last several days and end abruptly. the daily mean of the region was above 50 µg/m³ on eight days, around 2026-09-03, 2026-09-08, 2026-09-21 and 2026-10-13, and fell to 8 on 2026-11-09. the south stays below 35 except for one sharp episode from 2026-09-14 to 2026-09-16, and the center rarely leaves the range of 8 to 20.

the daily cycle has the same shape in every region. PM2.5 builds up through the night, peaks around six in the morning and drops to its minimum between late morning and mid afternoon, when the mixing layer is deepest. in the north the level at six in the morning is 2.3 times the level at noon. the heat map shows both patterns at once, dark bands at night that grow thicker during an episode.

counted once per grid cell, there were 499 runs of six hours or more above an AQI of 150, three quarters of them in the north. the longest lasted 70 hours, at Viet Tri from 2026-09-20 to 2026-09-23.

## weather as a driver

In [ ]:
drivers = {
    "wind_gusts_kmh": "wind\ngusts",
    "wind_speed_kmh": "wind\nspeed",
    "precipitation_mm": "rain",
    "cloud_cover_pct": "cloud\ncover",
    "temperature_c": "temperature",
    "pressure_msl_hpa": "pressure",
    "humidity_pct": "humidity",
}
corr = eda.correlations(hourly, "pm2_5_ugm3", list(drivers), "region", method="spearman")
corr = corr.loc[REGIONS]

fig, ax = plt.subplots(figsize=(8.0, 2.6))
image = ax.imshow(corr.to_numpy(), cmap=plots.DIVERGING, vmin=-0.6, vmax=0.6, aspect="auto")
for (row, column), value in np.ndenumerate(corr.to_numpy()):
    ax.text(column, row, f"{value:+.2f}", ha="center", va="center", fontsize=9, color=plots.INK)
ax.set_xticks(range(len(drivers)), list(drivers.values()))
ax.set_yticks(range(len(REGIONS)), REGIONS)
ax.tick_params(length=0)
ax.grid(False)
ax.spines[["left", "bottom"]].set_visible(False)
ax.set_title("rank correlation of hourly PM2.5 with the weather")
fig.colorbar(image, ax=ax, label="Spearman correlation", pad=0.02)
plots.save_figure(fig, FIGURES, "eda_weather_correlations")
plt.show()

In [ ]:
edges = [0, 3, 6, 9, 12, 15, 20, 50]
binned = eda.binned_mean(hourly, "wind_speed_kmh", "pm2_5_ugm3", edges, "region")
by_wind = binned.pivot(index="bin", columns="region", values="mean")[REGIONS]
positions = np.arange(len(by_wind))

fig, ax = plt.subplots()
for region in REGIONS:
    ax.plot(
        positions,
        by_wind[region],
        marker="o",
        markersize=7,
        markeredgecolor=plots.SURFACE,
        markeredgewidth=1.5,
        label=region,
        color=plots.REGION_COLORS[region],
    )
ax.set_xticks(positions, [f"{int(b.left)} to {int(b.right)}" for b in by_wind.index])
ax.set_ylim(0)
ax.set_title("mean PM2.5 by wind speed")
ax.set_xlabel("wind speed, km/h")
ax.set_ylabel(PM)
plots.legend_above(ax)
plots.save_figure(fig, FIGURES, "eda_pm2_5_by_wind_speed")
plt.show()
display(by_wind.round(1))

rain = np.where(hourly["precipitation_mm"] > 0.1, "wet hours", "dry hours")
display(hourly.groupby(["region", rain])["pm2_5_ugm3"].mean().unstack().loc[REGIONS].round(1))

In [ ]:
north_daily = (
    daily[daily["region"] == "north"]
    .groupby("day_local")[["pm2_5", "wind_speed", "precipitation"]]
    .mean()
)
panels = [("pm2_5", PM), ("wind_speed", "wind speed, km/h"), ("precipitation", "rain, mm per day")]

fig, axes = plt.subplots(3, 1, figsize=(8.0, 6.2), sharex=True)
for ax, (column, label) in zip(axes, panels):
    if column == "precipitation":
        ax.bar(north_daily.index, north_daily[column], width=0.7, color=plots.SERIES[0])
    else:
        ax.plot(north_daily.index, north_daily[column], color=plots.SERIES[0])
    ax.set_ylim(0)
    ax.set_ylabel(label)
axes[0].set_title("daily PM2.5, wind and rain in the north")
axes[-1].xaxis.set_major_formatter(DAY_FORMAT)
fig.align_ylabels(axes)
plots.save_figure(fig, FIGURES, "eda_north_pm2_5_wind_rain")
plt.show()
display(north_daily.corr()["pm2_5"].round(2).to_frame("correlation with daily PM2.5"))

wind is the strongest driver. hourly PM2.5 falls steadily with the wind speed in every region, in the north from 36 µg/m³ in light wind to 12 above 20 km/h. gusts relate to it more closely than the mean wind, with a rank correlation of -0.37 in the north and -0.47 in the south. rain cleans the air as well, wet hours are a third lower than dry hours in the north. humidity has a positive sign in the north and the south, which fits the calm and humid nights in which the particles build up.

on the daily scale the link is stronger. the daily mean PM2.5 of the north has a correlation of -0.64 with the daily wind speed and -0.42 with the daily rain. the three panels show the mechanism. every episode sits in a stretch of calm and dry days, and every clean spell follows wind or rain, most clearly the rain around 2026-09-14 and the strong wind of 2026-11-09.

## what this means for the forecast

In [ ]:
lags = list(range(1, 73))
memory = {
    "PM2.5": eda.autocorrelation(air, "pm2_5_ugm3", lags),
    "temperature": eda.autocorrelation(weather, "temperature_c", lags),
}

fig, ax = plt.subplots()
for (label, table), color in zip(memory.items(), plots.SERIES):
    ax.plot(table.index, table["median"], label=label, color=color)
    ax.fill_between(table.index, table["q25"], table["q75"], color=color, alpha=0.1, linewidth=0)
ax.axhline(0, color=plots.AXIS, linewidth=0.8)
plots.label_line_ends(ax)
ax.set_xticks(range(0, 73, 12))
ax.set_xlim(0, 82)
ax.set_title("autocorrelation of the hourly series")
ax.set_xlabel("lag, hours")
ax.set_ylabel("autocorrelation, median over points")
plots.legend_above(ax, ncols=2)
plots.save_figure(fig, FIGURES, "eda_autocorrelation")
plt.show()
at_lags = {name: table.loc[[1, 6, 12, 24, 48], "median"] for name, table in memory.items()}
display(pd.DataFrame(at_lags).round(2))

In [ ]:
evaluation = store.load(
    "forecast_evaluation",
    f"select target, horizon_h, model, mae, rmse, n, model_version from {HIVE}.curated_forecast_evaluation",
)
models = {
    "gbt": "gradient boosted trees",
    "persistence": "persistence",
    "same_hour_yesterday": "same hour yesterday",
    "moving_average_3h": "moving average 3 h",
    "moving_average_24h": "moving average 24 h",
}
targets = {"pm2_5": "PM2.5, MAE in µg/m³", "temperature": "temperature, MAE in °C"}
mae = evaluation.pivot_table(index=["target", "horizon_h"], columns="model", values="mae")
mae = mae[list(models)]
width = 0.15

fig, axes = plt.subplots(1, 2, figsize=(9.0, 3.8))
for ax, (target, title) in zip(axes, targets.items()):
    table = mae.loc[target]
    centers = np.arange(len(table))
    for index, (model, color) in enumerate(zip(models, plots.SERIES)):
        ax.bar(
            centers + (index - 2) * width,
            table[model],
            width=width * 0.85,
            color=color,
            label=models[model],
        )
    ax.set_xticks(centers, [f"{horizon} h ahead" for horizon in table.index])
    ax.set_title(title)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncols=5, bbox_to_anchor=(0.5, -0.08))
fig.suptitle("forecast error on the holdout days", x=0.01, ha="left", fontweight="bold")
fig.tight_layout()
plots.save_figure(fig, FIGURES, "eda_forecast_mae")
plt.show()
display(mae.round(3))

gain = pd.concat(
    {
        baseline: eda.skill_against(evaluation, "gbt", baseline).set_index(["target", "horizon_h"])["skill"]
        for baseline in list(models)[1:]
    },
    axis=1,
)
display(gain.round(3))

the gradient boosted trees beat every baseline one hour and six hours ahead, by 7 and 29 percent against persistence for PM2.5 and by 31 and 68 percent for temperature. one day ahead they lose 3 and 6 percent to persistence, which at that horizon is the value of the same hour yesterday.

the autocorrelation explains why that baseline is so strong. after six hours a series has moved to another part of its daily cycle and its correlation with the current value is 0.57 for PM2.5 and 0.25 for temperature, so a model that knows the hour of the day has a lot to add. after 24 hours the cycle is back where it started and the correlation is 0.57 and 0.87 again. the model has to rebuild the current value from tree splits and can only approximate it in steps, while persistence copies it exactly.

two changes follow for a later version of the model. the target can be the change from the current value instead of the level, so the model starts from the persistence forecast and only learns the correction. and the weather forecast for the target hour can become a feature, the previous section shows that wind and rain decide whether an episode builds up or ends, and the current features only describe the weather of the past 24 hours.

## the realtime view next to the batch view

the last table joins the two halves of the architecture in one query. the newest value of every point comes from Cassandra, where the speed layer writes it within a minute, and the mean of the same calendar month comes from the batch tables on HDFS.

In [ ]:
current = store.load(
    "current_vs_history",
    f"""
    select name, region, event_ts_utc, pm2_5_ugm3, aqi_computed, aqi_category,
           monthly_mean_pm2_5, pm2_5_above_mean
    from {VIEWS}.current_vs_history
    """,
)
print("realtime values of", current["event_ts_utc"].max(), "next to the batch mean of the month")
display(current.sort_values("pm2_5_above_mean", ascending=False).head(10))
display(current.groupby("region")[["pm2_5_ugm3", "monthly_mean_pm2_5"]].mean().loc[REGIONS].round(1))

when this extract was taken an episode was building up in the Red River delta. Hung Yen stood at 88 µg/m³, 59 above its mean of the month so far, and the north as a whole at 41 against a mean of 17, after two days of strong wind had cleaned the air. the center was at its mean and the south below it. the batch view alone would not show the episode before its next run, and the realtime view alone could not tell that these values are unusual for the month.

## takeaways

1. the pipeline delivers a complete hourly series for all 63 points, 108,612 rows per domain over 72 days, and the batch cleaning reduces the 136,512 raw rows of a domain to one row per event. the points hold 53 independent air quality series because of the grid of the CAMS model.
2. the north is the polluted region, with a mean PM2.5 of 30 µg/m³, three of four days above the WHO guideline and episodes of several days in the Red River delta.
3. PM2.5 follows a daily cycle in every region, with a peak near six in the morning and a minimum in the afternoon.
4. wind and rain control the episodes. calm and dry days build them and a windy or rainy day ends them.
5. the forecast model adds most at one and six hours. one day ahead it should predict the change from the current value and use the weather forecast.
6. the extract covers 2026-09 to 2026-11, so it shows the start of the dry season in the north but no full seasonal cycle. all values are model output, a comparison with station readings is still open.